# 05 - Baseline architecture
Find out which model the CAD2 Task 2 baseline uses, describe its architecture, and compare a few variants on the valid set.

**Before this, run `00_shared_drive_data.ipynb`:** the clone-Clarity cell and **Part A** (builds the valid-set dataset). It puts Clarity in `/content/clarity` and the dataset in `/content/cadenza_data/cad2/task2`.

Note: this notebook uses paths relative to `/content`, so a Colab session restart means running 00 again. Save results to Drive (`MyDrive/Cadenza Challenge/results/`) because free Colab can disconnect.

1. Check prerequisites (nothing is downloaded here)
Stops with a clear message if notebook 00 has not been run in this session.

In [1]:
import os
from pathlib import Path
os.chdir('/content')

root = Path('/content/cadenza_data/cad2/task2')
clarity_dir = Path('/content/clarity')
meta = root / 'metadata'
baseline = clarity_dir / 'recipes/cad2/task2/baseline'

need = {
    'Clarity baseline (run 00 clone cell or 01_setup)': baseline / 'enhance.py',
    'valid metadata (run 00 Part A)': meta / 'scenes.valid.json',
    'valid audio (run 00 Part A)': root / 'audio/valid',
}
missing = [k for k, p in need.items() if not p.exists()]
assert not missing, 'Missing: ' + '; '.join(missing)
print('prerequisites ok')

prerequisites ok


## Part A - Find the baseline model

2. Read the baseline recipe
Show `config.yaml` and `enhance.py`, then find where the model is loaded: which class, which file it comes from, and the causal / non-causal switch.

In [2]:
!grep -n "ConvTasNet\|from_pretrained\|import\|causal" /content/clarity/recipes/cad2/task2/baseline/enhance.py

!cat /content/clarity/recipes/cad2/task2/baseline/config.yaml

3:from __future__ import annotations
5:import json
6:import logging
7:from collections import Counter
8:from pathlib import Path
10:# pylint: disable=import-error
11:import hydra
12:import numpy as np
13:import torch
14:from numpy import ndarray
15:from omegaconf import DictConfig
16:from torchaudio.transforms import Fade
18:from clarity.enhancer.multiband_compressor import MultibandCompressor
19:from clarity.utils.audiogram import Listener
20:from clarity.utils.flac_encoder import read_flac_signal, save_flac_signal
21:from clarity.utils.source_separation_support import get_device
22:from recipes.cad2.task2.baseline.evaluate import (
28:from recipes.cad2.task2.ConvTasNet.local.tasnet import ConvTasNetStereo
128:    model: dict[str, ConvTasNetStereo],
175:    causality: str, device: torch.device, force_redownload: bool = True
176:) -> dict[str, ConvTasNetStereo]:
180:        causality (str): Causality of the model (causal or noncausal).
188:    causal = {"causal": "Causal", "noncausal":

3. List the models Cadenza published on HuggingFace
Which instruments have a model, and are there Causal as well as NonCausal versions?

In [3]:
# list models from the cadenzachallenge HuggingFace account
from huggingface_hub import list_models
for m in list_models(author='cadenzachallenge'):
    print(m.id)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


cadenzachallenge/ConvTasNet_Lyrics_NonCausal
cadenzachallenge/ConvTasNet_Lyrics_Causal
cadenzachallenge/ConvTasNet_Bassoon_NonCausal
cadenzachallenge/ConvTasNet_Cello_NonCausal
cadenzachallenge/ConvTasNet_Clarinet_NonCausal
cadenzachallenge/ConvTasNet_Flute_NonCausal
cadenzachallenge/ConvTasNet_Oboe_NonCausal
cadenzachallenge/ConvTasNet_Sax_NonCausal
cadenzachallenge/ConvTasNet_Viola_NonCausal
cadenzachallenge/ConvTasNet_Violin_NonCausal
cadenzachallenge/ConvTasNet_Bassoon_Causal
cadenzachallenge/ConvTasNet_Cello_Causal
cadenzachallenge/ConvTasNet_Clarinet_Causal
cadenzachallenge/ConvTasNet_Flute_Causal
cadenzachallenge/ConvTasNet_Oboe_Causal
cadenzachallenge/ConvTasNet_Sax_Causal
cadenzachallenge/ConvTasNet_Viola_Causal
cadenzachallenge/ConvTasNet_Violin_Causal
cadenzachallenge/Dynamic_Source_Separator_Causal
cadenzachallenge/Dynamic_Source_Separator_Noncausal
cadenzachallenge/CLIP2-BaselineMono
cadenzachallenge/CLIP2-BaselineBetterEar


4. Load one model and describe it
Print the structure, parameter count and config.

In [4]:

# TODO: load one pretrained model, print structure / parameter count / config

5. Architecture table
Fill in using the Conv-TasNet paper's notation, and compare with `third_party/Conv-TasNet/conv_tasnet.py`.

| | Cadenza baseline | naplab Conv-TasNet (third_party) |
|---|---|---|
| N (encoder filters) | | 512 |
| L (window) | | 2 ms |
| B (bottleneck) | | 128 |
| H (hidden) | | |
| P (kernel) | | |
| X (blocks per repeat) | | 8 |
| R (repeats) | | 3 |
| Causal | | |
| Sample rate | | 16 kHz |
| Mono / stereo | | |
| Outputs (sources) | | |
| Parameters | | |

6. Where the model sits in the pipeline
mixture -> one separator per instrument -> scene gains -> remix -> NAL-R + compressor (listener) -> output (32 kHz)

Notes: which parts are learned and which are fixed signal processing?

## Objective 2 - Reproduce the baseline separation step

What the baseline does for every scene (from `enhance.py`):
1. Cut the scene segment out of the mixture (44.1 kHz stereo).
2. For each source in the scene, run the pretrained `ConvTasNetStereo` for that instrument (`cadenzachallenge/ConvTasNet_<Instrument>_NonCausal`). Each model outputs **2 signals: target instrument + accompaniment** (everything else).
3. Keep `target + 0.1 * accompaniment` (`add_residual: 0.1` in `config.yaml`), apply gains, remix, compress for the listener.

Here we do steps 1-2 ourselves, save the separated stems (**PI1**) and score them against the reference stems with SDR (**PI2**). Nothing in Clarity is changed; we import its functions.

PI1-a. Import the baseline's own separation code
`separate_sources` is the exact function `enhance.py` uses (10 s chunks, 10% overlap, cross-fade).

In [5]:
import sys, json, time
import numpy as np, torch, soundfile as sf
from IPython.display import Audio, display

# Make Clarity's `recipes` package importable (Colab's torchtune ships another package called `recipes`)
sys.path.insert(0, str(clarity_dir))
for name in [m for m in sys.modules if m == 'recipes' or m.startswith('recipes.')]:
    del sys.modules[name]

from clarity.utils.flac_encoder import read_flac_signal
from recipes.cad2.task2.baseline.enhance import separate_sources
from recipes.cad2.task2.ConvTasNet.local.tasnet import ConvTasNetStereo

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cpu


PI1-b. Pick a validation scene
Same scene as notebook 03 (`S50001`), so the HAAQI score there can be linked to the SDR here.

In [6]:
scenes = json.load(open(meta / 'scenes.valid.json'))
music = json.load(open(meta / 'music.valid.json'))

scene_id = 'S50001'
scene = scenes[scene_id]
song = music[scene['music']]
print(scene_id, scene)
for key, entry in song.items():
    print(f"{key:9s} {entry['instrument']:10s} {entry['track']}  start={entry['start']} dur={entry['duration']}")

S50001 {'music': 'anitrasdance_001', 'gain': 'gain_0236'}
source_1  Cello      valid/anitrasdance_001/Cello.flac  start=0.0 dur=15
source_2  Viola      valid/anitrasdance_001/Viola.flac  start=0.0 dur=15
source_3  Violin_1   valid/anitrasdance_001/Violin_1.flac  start=0.0 dur=15
source_4  Violin_2   valid/anitrasdance_001/Violin_2.flac  start=0.0 dur=15
mixture   Mixture    valid/anitrasdance_001/mix_anitrasdance_001.flac  start=0.0 dur=15


PI1-c. Load the mixture and the reference stems
The last line checks the cut: if the stems add up to the mixture, the segment lines up.

In [7]:
SR = 44100   # input_sample_rate in baseline config.yaml

def load_clip(entry):
    '''Read a track and cut it to the scene segment (no cut if the file is already that long).'''
    signal, sr = read_flac_signal(root / 'audio' / entry['track'])
    assert sr == SR, f'expected {SR} Hz, got {sr}'
    n = int(entry['duration'] * sr)
    if len(signal) > n + 1:                      # full-length file -> cut the segment
        start = int(entry['start'] * sr)
        signal = signal[start:start + n]
    return signal                                 # (samples, channels)

mixture = load_clip(song['mixture'])
references = {k: load_clip(v) for k, v in song.items() if k != 'mixture'}
instruments = {k: v['instrument'].split('_')[0] for k, v in song.items() if k != 'mixture'}  # Violin_1 -> Violin, same as enhance.py

n = min(len(mixture), *(len(r) for r in references.values()))
mixture = mixture[:n]
references = {k: r[:n] for k, r in references.items()}

print('mixture:', mixture.shape, f'{n / SR:.1f}s')
print('sources:', instruments)
err = np.sum((mixture - sum(references.values())) ** 2) / np.sum(mixture ** 2)
print(f'stems vs mixture mismatch: {10 * np.log10(err + 1e-12):.1f} dB (very negative = aligned)')

mixture: (661500, 2) 15.0s
sources: {'source_1': 'Cello', 'source_2': 'Viola', 'source_3': 'Violin', 'source_4': 'Violin'}
stems vs mixture mismatch: -120.0 dB (very negative = aligned)


PI1-d. Separate with the pretrained models and save the stems
One model per instrument in the scene (two violins share the Violin model, like the baseline). Set `CAUSALITY = 'Causal'` and re-run to get the causal version for Comparison A.

In [8]:
CAUSALITY = 'NonCausal'      # baseline default (config.yaml: separator.causality = noncausal)
out_dir = Path(f'/content/exp_05_separation/{scene_id}_{CAUSALITY}')
out_dir.mkdir(parents=True, exist_ok=True)

estimates, accompaniments, timings = {}, {}, {}
for inst in sorted(set(instruments.values())):
    model_id = f'cadenzachallenge/ConvTasNet_{inst}_{CAUSALITY}'
    model = ConvTasNetStereo.from_pretrained(model_id).to(device).eval()
    t0 = time.time()
    # same call as decompose_signal() in enhance.py (default segment=10 s, overlap=0.1)
    est = separate_sources(model, mixture.T, sample_rate=SR, number_sources=2, device=device)
    timings[inst] = time.time() - t0
    target, accomp = est.squeeze(0).cpu().numpy()          # each (channels, samples)
    estimates[inst], accompaniments[inst] = target.T, accomp.T
    sf.write(out_dir / f'{inst}_target.flac', target.T, SR)
    sf.write(out_dir / f'{inst}_accompaniment.flac', accomp.T, SR)
    print(f'{model_id}: {timings[inst]:.1f}s on {device}')
    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

for key, ref in references.items():
    sf.write(out_dir / f'{key}_{instruments[key]}_reference.flac', ref, SR)
print('saved to', out_dir)
print(sorted(p.name for p in out_dir.iterdir()))

config.json:   0%|          | 0.00/204 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 26.4MB            

model.safetensors: downloading bytes:           |  0.00B            

cadenzachallenge/ConvTasNet_Cello_NonCausal: 57.3s on cpu


config.json:   0%|          | 0.00/204 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 26.4MB            

... (2 outputs left out)

config.json:   0%|          | 0.00/204 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 26.4MB            

model.safetensors: downloading bytes:           |  0.00B            

cadenzachallenge/ConvTasNet_Violin_NonCausal: 54.9s on cpu
saved to /content/exp_05_separation/S50001_NonCausal
['Cello_accompaniment.flac', 'Cello_target.flac', 'Viola_accompaniment.flac', 'Viola_target.flac', 'Violin_accompaniment.flac', 'Violin_target.flac', 'source_1_Cello_reference.flac', 'source_2_Viola_reference.flac', 'source_3_Violin_reference.flac', 'source_4_Violin_reference.flac']


PI1-e. Listen (first source only, to keep the notebook small): estimate, then reference

In [9]:
key = next(iter(references))
print(key, instruments[key])
display(Audio(estimates[instruments[key]].T, rate=SR))
display(Audio(references[key].T, rate=SR))

source_1 Cello


PI2-a. Install museval (BSSEval v4, the SDR used in SiSEC/MUSDB papers)

In [10]:
!pip install -q museval

PI2-b. Score the stems with SDR and save a CSV
Columns:
- `sdr_global`: whole-signal SDR, `10*log10(|s|^2 / |s - s_hat|^2)` (simple, the one the Cadenza papers quote)
- `sdr_global_residual0.1`: same, for `target + 0.1 * accompaniment` (what the baseline really remixes)
- `sdr_mixture_as_estimate`: SDR if we did no separation at all and used the mixture (the floor to beat)
- `museval_*`: BSSEval v4, median over 1 s windows (SDR, ISR, SIR, SAR)

In [11]:
import pandas as pd

def sdr_global(ref, est):
    '''Whole-signal SDR in dB.'''
    return 10 * np.log10((np.sum(ref ** 2) + 1e-10) / (np.sum((ref - est) ** 2) + 1e-10))

keys = list(references)
ref_arr = np.stack([references[k] for k in keys])                       # (sources, samples, channels)
est_arr = np.stack([estimates[instruments[k]] for k in keys])
res_arr = np.stack([estimates[instruments[k]] + 0.1 * accompaniments[instruments[k]] for k in keys])

try:
    import museval
    def bss(est):
        sdr, isr, sir, sar = museval.evaluate(ref_arr, est, win=SR, hop=SR)   # 1 s windows
        return [np.nanmedian(m, axis=1) for m in (sdr, isr, sir, sar)]
    t0 = time.time()
    bss_raw, bss_res = bss(est_arr), bss(res_arr)
    print(f'museval took {time.time() - t0:.0f}s')
except Exception as e:
    print('museval failed, only global SDR is reported:', repr(e))
    bss_raw = bss_res = None

rows = []
for i, k in enumerate(keys):
    row = {
        'scene': scene_id, 'song': scene['music'], 'source': k, 'instrument': instruments[k],
        'model': f'ConvTasNet_{instruments[k]}_{CAUSALITY}',
        'sdr_global': sdr_global(ref_arr[i], est_arr[i]),
        'sdr_global_residual0.1': sdr_global(ref_arr[i], res_arr[i]),
        'sdr_mixture_as_estimate': sdr_global(ref_arr[i], mixture),
        'separation_seconds': timings[instruments[k]],
    }
    if bss_raw is not None:
        row.update({'museval_sdr': bss_raw[0][i], 'museval_isr': bss_raw[1][i],
                    'museval_sir': bss_raw[2][i], 'museval_sar': bss_raw[3][i],
                    'museval_sdr_residual0.1': bss_res[0][i]})
    rows.append(row)

sdr_df = pd.DataFrame(rows)
sdr_df.to_csv(out_dir / 'sdr.csv', index=False)
print('saved', out_dir / 'sdr.csv')
display(sdr_df.round(2))

museval took 42s
saved /content/exp_05_separation/S50001_NonCausal/sdr.csv


,scene,song,source,instrument,model,sdr_global,sdr_global_residual0.1,sdr_mixture_as_estimate,separation_seconds,museval_sdr,museval_isr,museval_sir,museval_sar,museval_sdr_residual0.1
0,S50001,anitrasdance_001,source_1,Cello,ConvTasNet_Cello_NonCausal,15.60,14.82,2.670000,57.28,12.95,20.98,17.59,13.79,11.78
1,S50001,anitrasdance_001,source_2,Viola,ConvTasNet_Viola_NonCausal,3.62,0.93,-16.049999,55.90,3.17,5.73,12.50,3.10,1.75
2,S50001,anitrasdance_001,source_3,Violin,ConvTasNet_Violin_NonCausal,-6.62,-6.73,-12.020000,54.89,-7.79,8.21,-9.83,14.28,-7.82
3,S50001,anitrasdance_001,source_4,Violin,ConvTasNet_Violin_NonCausal,7.43,6.70,-4.380000,54.89,8.63,21.23,9.27,14.28,8.04


PI2-c. Copy the results to Drive (survives a Colab disconnect)

In [12]:
import shutil
drive_dir = Path('/content/drive/MyDrive/Cadenza Challenge/results/05_separation')
if Path('/content/drive/MyDrive').exists():
    shutil.copytree(out_dir, drive_dir / out_dir.name, dirs_exist_ok=True)
    print('copied to', drive_dir / out_dir.name)
else:
    print('Drive not mounted - run the first cell of notebook 00 to keep these results')

copied to /content/drive/MyDrive/Cadenza Challenge/results/05_separation/S50001_NonCausal


### Notes for the progress report (fill in after running)
- **Scene:** `S50001` - song: ___ - sources / instruments: ___
- **Model:** `ConvTasNetStereo` (`recipes/cad2/task2/ConvTasNet/local/tasnet.py`), pretrained `cadenzachallenge/ConvTasNet_<Instrument>_NonCausal`, 44.1 kHz stereo in, 2 outputs (target + accompaniment)
- **Separation settings:** 10 s chunks, 10% overlap (defaults of `separate_sources`; note `separation.segment: 6.0` in `config.yaml` is *not* passed by `decompose_signal`)
- **Device / time per model:** ___
- **SDR results** (paste the table): ___
- **Separation vs doing nothing:** `sdr_global` minus `sdr_mixture_as_estimate` = ___ dB
- **Effect of the 0.1 residual on SDR:** ___
- **What didn't work / surprises:** ___ (e.g. museval install, two voices of the same instrument getting the same estimate)

## Part B - Comparison numbers

7. Pick valid scenes
About 5 scenes, each with its first listener. Each scene takes roughly 4 min to enhance plus 1 min to evaluate.

In [13]:
# TODO: choose ~5 valid scenes and write a small scenes / scene_listeners file

8. Comparison A - NonCausal (default) vs Causal
HAAQI left / right / avg per scene. If there are no Causal models, compare against applying the gains to the whole mixture with no separation.

In [14]:
# TODO: run enhance.py + evaluate.py for each variant, collect scores.csv

9. Comparison B - separation quality
Per-stem SDR is now done in **Objective 2 - PI2** above. Re-run that section with `CAUSALITY = 'Causal'` for the causal numbers.

In [15]:
# TODO: per-stem SDR

10. Save results to Drive

In [16]:
# TODO: write the comparison CSVs to /content/drive/MyDrive/Cadenza Challenge/results/

## Summary
- Baseline model:
- Key differences from the original Conv-TasNet:
- Comparison results:
- Questions for Kirill:
- Next step (retraining): see `recipes/cad2/task2/ConvTasNet/` in Clarity (check that it exists)